# Bronze — Teams

Lê **todos** os CSVs de `landing/teams/` e escreve Delta Lake com tipagem explícita.

**Estratégia:** `overwrite` completo — teams é tabela de referência (8 linhas), idempotente.  
**Transformações:** cast de tipos, `_ingested_at`, `_source_layer`. Sem regras de negócio.

In [ ]:
# parameters
execution_date = "2025-06-15"
year = 2025
month = 6
day = 15

In [ ]:
import sys
import os

year, month, day = int(year), int(month), int(day)

notebooks_root = next(
    (p for p in ["/opt/airflow/notebooks", "/opt/spark/notebooks"]
     if os.path.exists(p)),
    "/opt/spark/notebooks",
)
if notebooks_root not in sys.path:
    sys.path.insert(0, notebooks_root)

from utils.spark_session import create_spark_session
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType, TimestampType
from pyspark.sql.utils import AnalysisException

In [ ]:
spark = create_spark_session("bronze_teams")

# Lê todos os CSVs de landing/teams (tabela de referência — full refresh)
input_path = "s3a://datalake/landing/teams/teams_*.csv"

try:
    df = spark.read.csv(input_path, header=True, inferSchema=False)
except AnalysisException:
    print(f"[bronze_teams] Nenhum arquivo encontrado em {input_path}. Encerrando.")
    spark.stop()
    df = None

if df is not None:
    df_bronze = (
        df.select(
            F.col("id").cast(IntegerType()).alias("id"),
            F.col("name").cast(StringType()).alias("name"),
            F.col("cost_center").cast(StringType()).alias("cost_center"),
            F.col("department").cast(StringType()).alias("department"),
            F.col("owner_email").cast(StringType()).alias("owner_email"),
            F.col("created_at").cast(TimestampType()).alias("created_at"),
        )
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_layer", F.lit("landing"))
    )

    output_path = "s3a://datalake/bronze/teams/"
    (
        df_bronze.write
        .format("delta")
        .mode("overwrite")
        .save(output_path)
    )

    print(f"[bronze_teams] {df_bronze.count()} linhas → {output_path}")
    df_bronze.printSchema()
    spark.stop()